<a href="https://colab.research.google.com/github/Ziqi-Li/GIS5106/blob/main/notebooks/W12_Segmentation_with_SAM_building_with_manual_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W12 Segmentation with SAM — Deep Learning for Feature Detection

This notebook introduces a deep learning workflow for detecting and extracting geospatial features from high-resolution imagery using the Segment Anything Model (SAM). Built on the cutting-edge transformer architecture by Meta AI, SAM enables flexible and high-accuracy object segmentation guided by user prompts or automated cues. In this lab, we apply SAM through its geospatial wrapper `segment-geospatial` to demonstrate how modern deep learning models can support mapping tasks such as identifying vegetation, rooftops, water bodies, or other landscape features. This project is inspired by the [segment-geospatial project](https://github.com/opengeos/segment-geospatial) and builds on Meta AI’s [Segment Anything](https://segment-anything.com) model.

In this example, you will learn how deep learning can be embedded into spatial workflows, how outputs can be transformed into GIS-ready formats like GeoJSON, and how these processes may support real-world applications like environmental monitoring, urban planning, and more.

This example is based on materials from Prof. Bo Zhao at University of Washington with some tweaks.



### Objectives

By the end of this example, you should be able to:

- Understand how deep learning supports object segmentation in geographic imagery;
- Use the Segment Anything Model (SAM) to extract features from aerial/satellite imagery;
- Convert raster segmentation into vector data formats (e.g., GeoJSON);
- Visualize results in interactive web maps using `leafmap`;
- Reflect critically on the promises and limitations of using AI in geographic contexts.


### Workflow

This notebook follows the steps below:

1. **Import Required Libraries**  
   Load `leafmap`, `segment-geospatial`, and other packages needed for mapping, segmentation, and visualization.

2. **Acquire Imagery Data**  
   Use an interactive map to select a Region of Interest (ROI), then download aerial imagery tiles as a GeoTIFF file using `leafmap`.

3. **Initialize the SAM Model**  
   Set up the `SamGeo2` deep learning segmentation model with the downloaded imagery.

4. **Load Prompt Boxes (Optional)**  
   Load external vector data (e.g., bounding boxes from GeoJSON) to guide SAM in identifying specific buildings.

5. **Segment Building footprint**  
   Use SAM to generate a raster mask showing where buildings has been identified.

6. **Visualize and Export Results**  
   Overlay the original imagery, segmentation mask, and vector polygons on an interactive map. Export results in GeoTIFF and GeoJSON formats for further use in GIS applications.

## Step 1: Install and Import Required Dependencies

To begin, make sure all necessary packages are installed in your environment. If you're running this notebook in Google Colab or another clean environment, you may need to uncomment and run the following cell:


In [ ]:
# Uncomment and run if needed
%pip install segment-geospatial leafmap geopandas localtileserver segment-geospatial[samgeo2]

Now that the dependencies are installed, let's import the necessary Python libraries:

- `leafmap`: for creating interactive maps and retrieving satellite imagery;
- `samgeo.SamGeo2`: a wrapper for using the Segment Anything Model (SAM) on geospatial imagery.

In [ ]:
import leafmap
from samgeo import SamGeo2

## Step 2: Acquire or Prepare Geospatial Data

We start by creating an interactive map using `leafmap`. This map allows you to select a Region of Interest (ROI) directly from satellite imagery.

- The map is centered at a location on FSU Campus (you can modify it).
- A high-resolution SATELLITE basemap is added for visual reference.
- The ROI you select will later be used for downloading imagery and running SAM-based segmentation.

In [ ]:
# Initialize an interactive map centered at FSU Campus
m = leafmap.Map(
    center=[30.441394024414933, -84.29811573285318],
    zoom=17,
    height="800px"
)

# Add a satellite basemap for visual ROI selection
m.add_basemap("SATELLITE")

# Display the interactive map
m

## Step 2: Select an Area of Interest (ROI)

Instead of selecting a region interactively in the notebook, you can also define a custom Region of Interest (ROI) using [geojson.io](https://geojson.io):

1. Visit [https://geojson.io](https://geojson.io)
2. Use the drawing tools to create a rectangle or polygon
3. Copy the coordinates from the exported GeoJSON
4. Assign the bounding box (bbox) manually in your notebook

> 🧭 This method is helpful when working with predefined study areas or collaborating across platforms.

In [ ]:
# Set the output file name for the downloaded GeoTIFF
image = "Image.tif"

# Download high-resolution satellite tiles and save them as a single GeoTIFF
leafmap.map_tiles_to_geotiff(
    output=image,       # Output filename
    bbox = [-84.29809675855033, 30.443524831567117, -84.29320586964847, 30.439618789531707],# The bounding box from user
    zoom=19,
    source="Satellite",
    overwrite=True
)

After downloading satellite tiles as a GeoTIFF raster, we now overlay it on the interactive map for visual inspection.

- The previous satellite basemap is hidden to avoid overlap.
- The newly downloaded high-resolution image is added as a custom raster layer.

This step confirms the imagery has been correctly downloaded and georeferenced before applying segmentation. The imagery will serve as the input for deep learning–based segmentation in the next step.

In [ ]:
# Hide the original satellite basemap to avoid visual overlap
m.layers[-1].visible = False

# Add the downloaded GeoTIFF image as a new raster layer
m.add_raster(image, layer_name="Image")

# Display the updated map with the new imagery
m

## Step 3: Initialize the Segment Anything Model (SAM)


We now initialize the Segment Anything Model (SAM) using the `SamGeo2` class from the `segment-geospatial` library.

- **`model_id="sam2-hiera-large"`** loads the latest version of SAM 2 with the hierarchical transformer backbone for high-resolution inference.
- **`automatic=False`** means segmentation will be guided by user prompts (e.g., points, boxes) instead of automatic segmentation.

> This step loads the deep learning model and prepares it to process the satellite image. You can modify the model ID to test different versions of SAM (e.g., `sam-vit-h`, `sam2-b`, etc.).

In [ ]:
# Initialize the Segment Anything model with a high-resolution hierarchical backbone
sam = SamGeo2(
    model_id="sam2-hiera-large",  # SAM 2 model with large hierarchical transformer
    automatic=False                # Use manual prompts instead of automatic segmentation
)

In [ ]:
# Load the GeoTIFF image into the SAM model for segmentation
sam.set_image(image)

We can provide bounding boxes from an existing vector dataset to guide the SAM model. These boxes act as spatial prompts, helping the model focus on specific areas of interest.

In this example, we use an available GeoJSON file containing bounding boxes of buildings: https://raw.githubusercontent.com/Ziqi-Li/GIS5106/refs/heads/main/data/parking_lots.geojson

These bounding boxes were created using [geojson.io](https://geojson.io), a free and simple tool for drawing and exporting spatial features.

> 💡 This approach enables SAM to perform segmentation tasks informed by existing spatial annotations — perfect for workflows that combine human-curated knowledge with AI-driven automation.


In [ ]:
# Load an existing GeoJSON file that contains bounding boxes for buildings
geojson = "https://raw.githubusercontent.com/Ziqi-Li/GIS5106/refs/heads/main/data/parking_lots.geojson"

To visually confirm the spatial relationship between the downloaded image and the external vector prompts, we overlay both on a unified interactive map.

- The raster image is the high-resolution satellite tile we downloaded earlier.
- The vector layer consists of bounding boxes that serve as prompts for the segmentation model.
- The boxes are styled for clarity with yellow outlines.

> This visual check ensures that the image and vector inputs are correctly aligned before applying deep learning.

In [ ]:
# Create a fresh interactive map
m = leafmap.Map()

# Add the downloaded satellite imagery (GeoTIFF)
m.add_raster(image, layer_name="image")

# Define styling for bounding boxes
style = {
    "color": "#ffff00",      # Yellow outline
    "weight": 2,             # Line thickness
    "fillColor": "#7c4185",  # Fill color (used here as transparent)
    "fillOpacity": 0         # No fill opacity
}

# Overlay bounding boxes from external GeoJSON
m.add_vector(
    geojson,
    style=style,
    zoom_to_layer=True,      # Automatically zoom to fit the boxes
    layer_name="Bounding boxes",
    info_mode=None           # Disable info popups
)

# Display the map
m

## Step 4: Predict and Visualize the Segmentation Mask

With the bounding boxes now aligned on the raster image, we use them as segmentation prompts for the SAM model.

- The `sam.predict()` function applies deep learning–based segmentation to each box region.
- The result is saved as a raster mask (`.tif`) where segmented areas are assigned pixel values.
- The prediction respects spatial reference (`EPSG:4326`) and saves output in `uint8` format for efficiency.

> This step shows how a deep learning model can use existing spatial knowledge (vector prompts) to extract features from remote sensing imagery.

In [ ]:
# Define the output filename for the segmentation mask
output_masks = "mask2.tif"

# Use SAM to perform segmentation using bounding box prompts from the GeoJSON
sam.predict(
    boxes=geojson,          # Vector-based bounding boxes as prompts
    point_crs="EPSG:4326",  # Coordinate reference system of the input GeoJSON
    output=output_masks,    # Filepath to save the output mask
    dtype="uint8"           # Data type of the output raster
)

After running the SAM model, the output is saved as a raster mask (GeoTIFF) where segmented areas (e.g., buildings) are highlighted.

We now overlay this result on the interactive map to:

- Compare the segmentation output with the original satellite image;
- Assess the accuracy and alignment of the deep learning model's predictions.

> 🌱 The segmented mask is shown with partial transparency (`opacity=0.5`) for visual blending with the base imagery.

In [ ]:
# Overlay the segmentation result (raster mask) on the interactive map
m.add_raster(
    output_masks,      # Path to the SAM output mask (GeoTIFF)
    nodata=0,          # Pixels with value 0 are treated as transparent
    opacity=0.5,       # Set transparency so we can see the base image underneath
    layer_name="Building masks"  # Legend label for this layer
)

m # Display the map

## Step 5: Post-Processing and Export Results

You can use the `region_groups()` method to clean up the segmentation results, such as removing small regions, and filling holes. In addition, you can compute geometric properties of the regions, such as area, perimeter, eccentricity, and solidity.

In [ ]:
out_image = "building_masks.tif"
out_vector = "building_vector.geojson"
array, gdf = sam.region_groups(
    output_masks, min_size=50, out_vector=out_vector, out_image=out_image
)

The predicted mask is processed into grouped regions based on connectivity and size. Smaller regions are filtered out, and the result is saved as both raster and vector formats.

In [ ]:
gdf.head()

In [ ]:
gdf.plot()

This interactive map brings together multiple layers of spatial data for visual inspection and interpretation.

In [ ]:
# Create a new interactive map
m = leafmap.Map()

# Add the original high-resolution satellite image layer
m.add_raster(image, layer_name="Image")

# Define custom styling for the vector polygons (building boundaries)
style = {
    "color": "#ffff00",      # Yellow border color for vector polygons
    "weight": 2,             # Border thickness
    "fillColor": "#7c4185",  # Fill color (not used visually here)
    "fillOpacity": 0         # Transparent fill for vector display
}

# Add the raster mask output from SAM model (segmented building regions)
m.add_raster(
    out_image,
    colormap="tab20",     # Use categorical colormap for region IDs
    nodata=0,             # Mask pixels with 0 are treated as transparent
    opacity=0.7,          # Semi-transparent overlay
    layer_name="Building masks"
)

# Add the vectorized polygons extracted from the raster mask
m.add_vector(
    out_vector,
    style=style,              # Styled in yellow outlines
    zoom_to_layer=True,       # Automatically zoom to fit this layer
    layer_name="Building vector"
)

# Add the original bounding box prompts used to guide segmentation
m.add_vector(
    geojson,
    style={"color": "blue", "fillOpacity": 0},  # Blue outline for prompt boxes
    layer_name="Bounding boxes",
    info_mode=None           # Disable pop-up on hover
)

# Enable interactive layer control for toggling visibility
m.add_layer_manager()

# Display the map
m

To evaluate the quality and alignment of the deep learning–based segmentation, we use a **split-map viewer** that allows side-by-side comparison:

- **Left**: The SAM model's predicted building mask.
- **Right**: The original high-resolution aerial imagery.

You can drag the vertical slider to dynamically compare what the model sees vs. the raw image.

> 🔍 This tool helps assess model accuracy, over/under-segmentation, and alignment with visible landscape features.

In [ ]:
# Create a new interactive map
m = leafmap.Map()

# Add the base satellite image layer for reference
m.add_raster(image, layer_name="Image")

# Use split_map() to create a swipe (side-by-side) comparison
m.split_map(
    out_image,      # Left side: raster mask output from SAM
    image,          # Right side: original satellite imagery
    left_label="Building masks",         # Label for left side
    right_label="Aerial imagery",    # Label for right side
    left_args={
        "colormap": "tab20",         # Use categorical colormap for mask regions
        "nodata": 0,                 # Treat value 0 as transparent
        "opacity": 0.7               # Set left layer to be semi-transparent
    }
)

# Display the swipe-enabled map for visual comparison
m

In this execise, we explored how deep learning can be applied to geospatial data using the **Segment Anything Model (SAM)**. Starting from satellite imagery, we walked through a complete SAM pipeline — from selecting a region of interest, loading prompt geometries, running segmentation, to visualizing and exporting results.

Through this process, you have:

- Learned how to integrate spatial data with foundation vision models;
- Applied vector prompts to guide object segmentation;
- Visualized and compared results using interactive web maps;
- Exported raster and vector data for further GIS-based analysis.


## Step 6: Critical Reflection: Beyond Segmentation

While this lab demonstrates the technical capabilities of SAM for geospatial feature extraction, it is important to recognize the broader implications of automated segmentation in geography.

The boundaries produced by the model are shaped by the prompts provided — often drawn by users or pre-existing datasets. These prompts inevitably carry assumptions about space, scale, and meaning, which may reflect dominant perspectives while excluding local or alternative knowledges.

Automated segmentation offers speed and scalability but often at the cost of nuance. It may struggle with ambiguous features, mixed land use, or regions undergoing rapid change, where rigid classification fails to capture lived spatial complexity.

The outputs of deep learning models can appear authoritative, but they require careful evaluation. Their performance may vary across geographic regions, imagery quality, or cultural landscapes. Over-reliance on AI-generated results risks obscuring these spatial and social variances.

The environmental cost of deploying large-scale models should also be considered. Training and running models like SAM involves substantial computational power, contributing to energy consumption and resource use with global ecological consequences.
